# Setting Up the database and graph

BookRank makes use of sqlite for storing the books and ratings metadata, and igraph for creating the graph.

Some of the code cells below require a lot of memory. You are adviced to run the code below on a system with 16GB RAM or more.

The code below is designed for goodbooks-10k and goodbooks-10k-extended. To use this code with other, larger datasets (like UCSD book graph), more memory and storage may be needed.

My advice is to run the code cells until you reach the section "Setting up the graph". Once you reach that section, restart the kernel (and hence clear memory), and run the cells from there.

In [1]:
import sqlite3
import pandas as pd

In [2]:
b = pd.read_csv("res/books_enriched.csv")

In [ ]:
# for each book in the dataset, we normalize its average goodreads rating to the global dataset average goodreads rating,
# centered on 3.
# S - average goodreads rating for a book
# C - global dataset average goodreads rating

def get_avg_weighted_score(S, C):
    if S > C:
        ws = 3 + (S - C)*(2 / (5 - C))
    else:
        ws = 3 + (S - C)*(2 / (C - 1))
    
    return ws

In [ ]:
C = b["average_rating"].mean()
print(f"Avg rating: {C}")

### Setting up the database and populating the tables

In [ ]:
conn = sqlite3.connect("data.db")
curr = conn.cursor()

query = """
    CREATE TABLE IF NOT EXISTS books (
        bookId INTEGER PRIMARY KEY,
        goodreadsBookId INTEGER,
        isbn INTEGER,
        authors TEXT,
        title TEXT,
        ratingsCount INTEGER,
        averageRating REAL,
        weightedScore REAL
    );

    CREATE TABLE IF NOT EXISTS ratings (
        bookIdA INTEGER,
        bookIdB INTEGER,
        averageRating REAL,
        sumRating INTEGER,
        ratingsCount INTGER,
        weightedScore REAL,
        PRIMARY KEY(bookIdA, bookIdB)
    );

    CREATE TABLE IF NOT EXISTS genres (
        genreId INTEGER PRIMARY KEY,
        name TEXT
    );

    CREATE TABLE IF NOT EXISTS book_genre (
        bookId INTEGER,
        genreId INTEGER,
        FOREIGN KEY (bookId) REFERENCES books(bookId),
        FOREIGN KEY (genreId) REFERENCES genres(genreId)
    );
"""

curr = curr.executescript(query)

conn.commit()
conn.close()

#### Populating the tables with data: books

In [ ]:
books = []

for row in b.itertuples():
    book = {
        "bookId": row.book_id,
        "goodreadsBookId": row.goodreads_book_id,
        "internalId": None,
        "isbn": row.isbn,
        "authors": row.authors,
        "title": row.title,
        "averageRating": row.average_rating,
        "weightedScore": get_avg_weighted_score(row.average_rating, C),
        "ratingsCount": row.ratings_count
    }
    books.append(book)

In [ ]:
conn = sqlite3.connect("data.db")

curr = conn.cursor()

query = """
    INSERT INTO books VALUES(:bookId, :goodreadsBookId, :isbn, :authors, :title, :ratingsCount, :averageRating, :weightedScore)
"""

curr.executemany(query, books)

conn.commit()
conn.close()

#### Populating the tables with data: ratings

The code below may take a while to run. There are 53424 users to process in total (for goodbooks-10k).

In [ ]:
r = pd.read_csv( 'res/ratings.csv' )

In [ ]:
users = {
    uid: group.set_index('book_id')['rating'].to_dict()
    for uid, group in r.groupby('user_id')
}

In [ ]:
from itertools import permutations

# this might take a while

ratings = {}

for userId, userRatings in  users.items():
    print(f"\rUserId: {userId}", end='', flush=True)
    for bookIdA, bookIdB in permutations(userRatings.keys(), 2):
        # first add to A -> B
        if (bookIdA, bookIdB,) not in ratings:
            ratings[(bookIdA, bookIdB,)] = {
                "averageRating": 0,
                "sumRating": 0,
                "ratingsCount": 0,
                "weightedScore": 0
            }
        ratings[(bookIdA, bookIdB,)]["sumRating"] += userRatings[bookIdB]
        ratings[(bookIdA, bookIdB,)]["ratingsCount"] += 1

        # then B -> A
        if (bookIdB, bookIdA,) not in ratings:
            ratings[(bookIdB, bookIdA,)] = {
                "averageRating": 0,
                "sumRating": 0,
                "ratingsCount": 0,
                "weightedScore": 0
            }
        ratings[(bookIdB, bookIdA,)]["sumRating"] += userRatings[bookIdA]
        ratings[(bookIdB, bookIdA,)]["ratingsCount"] += 1

In [ ]:
for abPair, rating in ratings.items():
    rating["averageRating"] = rating["sumRating"] / rating["ratingsCount"]
    rating["weightedScore"] = get_avg_weighted_score(rating["averageRating"], C)

In [ ]:
conn = sqlite3.connect("data.db")

curr = conn.cursor()

dataGen = (
    (k[0], k[1], *v.values()) for k, v in ratings.items()
)

query = """
    INSERT INTO ratings VALUES(?, ?, ?, ?, ?, ?)
"""

curr.executemany(query, dataGen)

conn.commit()
conn.close()

#### Populating the tables with data: genres

In [22]:
bookGenres = [set() for i in range(10001)]
genreIds = {}
idsGenre = []
genreIdCount = 0

for row in b.itertuples(index=False):
    genres = eval(row.genres) #make into list
    for genre in genres:
        if genre not in genreIds:
            genreIds[genre] = genreIdCount
            idsGenre.append(genre)
            genreIdCount += 1
        bookGenres[row.book_id].add(genreIds[genre])

In [ ]:
idsGenreTuples = []

for i, genre in enumerate(idsGenre):
    idsGenreTuples.append((i, genre,))

idsGenreTuples = tuple(idsGenreTuples)

In [ ]:
conn = sqlite3.connect("data.db")

query = "INSERT INTO genres VALUES (?, ?)"

curr = conn.cursor()
curr.executemany(query, idsGenreTuples)

conn.commit()
conn.close()

#### Populating the tables with data: book_genre

In [ ]:
bookId_genreId = []
for bookId in range(1, 10001):
    genres = bookGenres[bookId]
    for genreId in genres:
        bookId_genreId.append((bookId, genreId,))
    

In [ ]:
conn = sqlite3.connect("data.db")

query = "INSERT INTO book_genre VALUES (?, ?)"

curr = conn.cursor()
curr.executemany(query, bookId_genreId)

conn.commit()
conn.close()

The database should now be set up properly. You can test it by running the code cell below.

In [25]:
conn = sqlite3.connect("data.db")

query = """
    SELECT 
        books.title, 
        books.bookId, 
        GROUP_CONCAT(genres.name, ', ') AS all_genres
    FROM books 
    JOIN book_genre ON books.bookId = book_genre.bookId 
    JOIN genres ON book_genre.genreId = genres.genreId
    WHERE books.bookId = ?
    GROUP BY books.bookId
"""

curr = conn.cursor()
curr.execute(query, (1,))
print(curr.fetchall())

conn.close()

[('The Hunger Games (The Hunger Games, #1)', 1, 'young-adult, fiction, fantasy, science-fiction, romance')]


## Setting up the graph

You may want to restart the python notebook kernel and continue from here (to clear the memory).

In [1]:
import igraph as ig
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect("data.db")

edges = pd.read_sql("SELECT * FROM ratings", conn)


conn.close()

In [3]:
conn = sqlite3.connect("data.db")

vertices = pd.read_sql("SELECT bookId, goodreadsBookId, weightedScore, ratingsCount FROM books", conn)

conn.close()

In [14]:
g = ig.Graph.DataFrame(edges, directed=True)

In [16]:
# delete the 0th node (orphan node) since bookIds are 1-indexed but igraph is 0-indexed
g.delete_vertices([0])

In [18]:
# now assign the attributes for each node
g.vs["bookId"] = vertices["bookId"].to_list()
g.vs["goodreadsBookId"] = vertices["goodreadsBookId"].to_list()
g.vs["weightedScore"] = vertices["weightedScore"].to_list()
g.vs["ratingsCount"] = vertices["ratingsCount"].to_list()

In [ ]:
#assign the genres to each node

df = pd.read_csv("res/books_enriched.csv")

bookGenres = [set() for i in range(10000)]
genreIds = {}
idsGenre = []
genreIdCount = 0

for row in df.itertuples(index=False):
    genres = eval(row.genres) #make into list
    for genre in genres:
        if genre not in genreIds:
            genreIds[genre] = genreIdCount
            idsGenre.append(genre)
            genreIdCount += 1
        bookGenres[row.book_id-1].add(genreIds[genre])

In [22]:
g.vs["genres"] = bookGenres

Graph has been created. Now we save the graph as a pickle file. Other formats like ```.gml``` or ```.graphml``` do not support sets data structures,
which BookRank uses to store genres. They also take up more space. Only use those formats if you want to view the graph in other software.

In [27]:
g.save("graph.pkl", format="pickle")